# PaddleOCR Vietnamese Training
## Optimized for >90% accuracy

Based on production-proven workflow

In [ ]:
# Cell 1: Check GPU & Install Paddle 3.2.0
!nvidia-smi

# Install Paddle GPU 3.2.0 (match CUDA 12.6)
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

In [ ]:
# Cell 2: Verify Paddle & Clone PaddleOCR
import paddle

print("Paddle version:", paddle.__version__)
print("CUDA available:", paddle.device.cuda.device_count())
print("Compiled with CUDA:", paddle.is_compiled_with_cuda())

# Clone PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

# Install requirements (NO paddlex to avoid conflicts)
!pip install -r PaddleOCR/requirements.txt --no-deps

# Install specific versions (CRITICAL)
!pip install --force-reinstall --no-cache-dir \
    numpy==1.26.4 \
    scipy==1.11.4 \
    scikit-learn==1.3.2 \
    opencv-python-headless \
    shapely \
    pyclipper \
    lmdb \
    tqdm \
    visualdl \
    rapidfuzz \
    PyYAML \
    pillow

In [ ]:
# Cell 3: Clone project
%cd /kaggle/working
!git clone https://github.com/YOUR_USERNAME/paddleocr-v5-vietnamese.git
%cd paddleocr-v5-vietnamese

# Download pretrained model
!wget -q https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams \
    -O pretrain_models/latin_PP-OCRv5_mobile_rec_pretrained.pdparams

!ls -lh pretrain_models/

In [ ]:
# Cell 4: Prepare data
!python prepare_data.py \
    --input_dir /kaggle/input/datasets/sirimiriiii13/vocr-rec/FinalData \
    --output_dir ./data \
    --fix_config config.yml \
    --base_dir /kaggle/working/paddleocr-v5-vietnamese

# Verify
!echo "\n📊 Data summary:"
!wc -l data/train_list.txt data/val_list.txt
!echo "\n📄 Sample lines:"
!head -3 data/train_list.txt

In [ ]:
# Cell 5: Train
!bash train.sh

# Or manual:
# !cd PaddleOCR && python -m paddle.distributed.launch \
#     --gpus '0,1' \
#     tools/train.py \
#     -c /kaggle/working/paddleocr-v5-vietnamese/config.yml

In [ ]:
# Cell 6: Check results
!tail -50 logs/train_*.log | grep -i "acc:" | tail -10

In [ ]:
# Cell 7: Test model
!cd PaddleOCR && python tools/infer_rec.py \
    -c /kaggle/working/paddleocr-v5-vietnamese/config.yml \
    -o Global.checkpoints=/kaggle/working/paddleocr-v5-vietnamese/output/vi_ppocr_v5/best_accuracy \
       Global.infer_img=/kaggle/working/paddleocr-v5-vietnamese/data/val_data \
       Global.save_res_path=/kaggle/working/results.txt \
       Global.use_gpu=True

# View results
!echo "\n📝 Results:"
!head -20 /kaggle/working/results.txt

In [ ]:
# Cell 8: Export model
!cd PaddleOCR && python tools/export_model.py \
    -c /kaggle/working/paddleocr-v5-vietnamese/config.yml \
    -o Global.pretrained_model=/kaggle/working/paddleocr-v5-vietnamese/output/vi_ppocr_v5/best_accuracy \
       Global.save_inference_dir=/kaggle/working/inference

!ls -lh /kaggle/working/inference/

In [ ]:
# Cell 9: Package for download
!tar -czf /kaggle/working/vietnamese_ocr_model.tar.gz \
    inference/ \
    dict/vi_dict.txt \
    config.yml

!ls -lh /kaggle/working/vietnamese_ocr_model.tar.gz